<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Quantize_LLMs_to_NVFP4_with_LLM_Compressor_Calibration_with_Long_Sequences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*More details in this article: [How Good is NVFP4 for Reasoning and with Small Models?](https://kaitchup.substack.com/p/how-good-is-nvfp4-for-reasoning-and)*

This notebook shows how to quantize LLMs with NVFP4, using LLM Compressor.
For the calibration steps, long sequences are used. In the code below, the samples are from open-r1/OpenR1-Math-220k, using sequences of at least 16k tokens.

In the article, I found that this might not be ideal. Here are some ideas of improvements:
* use 1024 samples
* 512 samples shorter than 16k tokens
* 512 samples longer than 16k tokens

# Installation

In [ ]:
!pip install llmcompressor datasets transformers

# Weight and Activation Quantization

In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

from datasets import load_dataset
NUM_CALIBRATION_SAMPLES=512
MAX_SEQUENCE_LENGTH=32000
# Load dataset.


TOKEN_THRESHOLD = 16000

# Load full split (filter first, then optionally downsample)
ds = load_dataset("open-r1/OpenR1-Math-220k", split=f"train")


def add_len(example):
    # UltraChat has a 'messages' list of {role, content}
    if "messages" in example and isinstance(example["messages"], list):
        # Use the model's chat template if available for accurate token counts
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    else:
        # Fallbacks for other schemas
        text = example.get("text") or (
            f"{example.get('prompt','')}\n{example.get('response','')}"
        )
    example["n_tokens"] = len(tokenizer(text, add_special_tokens=True).input_ids)
    return example

# Compute token lengths (parallelize if you have multiple CPUs)
ds = ds.map(add_len, num_proc=4, desc="Computing token lengths")

# Keep only very long samples
ds = ds.filter(lambda ex: ex["n_tokens"] >= TOKEN_THRESHOLD)

# Now shuffle and (optionally) cap to your calibration budget
ds = ds.shuffle(seed=42)
if "NUM_CALIBRATION_SAMPLES" in globals():
    ds = ds.select(range(min(NUM_CALIBRATION_SAMPLES, len(ds))))



# Preprocess the data into the format the model is trained with.
def preprocess(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False,)}
ds = ds.map(preprocess)

# Tokenize the data (be careful with bos tokens - we need add_special_tokens=False since the chat_template already added it).
def tokenize(sample):
    return tokenizer(sample["text"], padding=False, max_length=MAX_SEQUENCE_LENGTH, truncation=True, add_special_tokens=False)
ds = ds.map(tokenize, remove_columns=ds.column_names)

# Configure the quantization algorithm to run.
recipe = QuantizationModifier(targets="Linear", scheme="NVFP4", ignore=["lm_head"])

# Apply quantization.
oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

# Save to disk compressed.
SAVE_DIR = MODEL_ID.rstrip("/").split("/")[-1] + "-2klen-NVFP4"
model.save_pretrained(SAVE_DIR, save_compressed=True)
tokenizer.save_pretrained(SAVE_DIR)